# Step 1: 激活离群点（Emergent Outliers）

**目标**：用 forward hook 抓 LLM 每个 `Linear` 的输入激活，亲眼看到 **emergent outliers**——少数 channel 的幅值是均值的 **20×+**、且跨 token 结构性持续。这是 LLM 量化难的根因，也是后续 SmoothQuant/AWQ/LLM.int8() 要解决的同一现象。

**对应 OUTLINE 课时**：1.1 激活离群点（~50 分钟）。

> 本模块是**原理层**：从零用 PyTorch 实现量化数学（不调 llm-compressor）。s1 用合成数据 + 真 Qwen2.5 验证。

In [ ]:
%%capture
import math, json, pathlib
import torch
import torch.nn as nn
import ipytest
ipytest.autoconfig()

In [ ]:
# Setup cell：notebook 向上发现模块根（含 steps/ + pyproject.toml），绝不依赖裸相对路径。
# 规范见 course/NOTEBOOK_CONVENTIONS.md 第 2 节。所有文件路径从 MODULE_ROOT 派生。
def _find_module_root(start):
    p = pathlib.Path(start).resolve()
    for cand in [p, *p.parents]:
        if (cand / "steps").is_dir() and (cand / "pyproject.toml").exists():
            return cand
    raise RuntimeError("找不到模块根（含 steps/ + pyproject.toml）；请在模块目录内 cd course/m1-activation-outliers 启动 jupyter")

MODULE_ROOT    = _find_module_root(pathlib.Path.cwd())
MODEL_DIR      = MODULE_ROOT / "models" / "Qwen2.5-7B-Instruct"      # 与 scripts/download_model.sh 一致
TINY_MODEL_DIR = MODULE_ROOT / "models" / "Qwen2.5-0.5B-Instruct"     # L3 先在 0.5B 上验，再上 7B
OUT_ROOT       = MODULE_ROOT / "out"                                 # 已 gitignore
OUT_ROOT.mkdir(parents=True, exist_ok=True)
print("MODULE_ROOT:", MODULE_ROOT)
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability()
    print("GPU:", torch.cuda.get_device_name(), "（sm{}{}，cap={}）".format(cap[0], cap[1], cap),
          "— 支持 FP8" if cap >= (8, 9) else "")
else:
    print("无 GPU（仅 L1/L2 可跑；L3 自动跳过）")

## 原理：为什么 LLM 量化变难了

小模型（GPT-2 量级）的激活分布近似高斯，绝大多数 channel 幅值集中在均值附近——朴素 per-tensor INT8 量化（用一个全局 scale）就够。但 **6B 以上的 LLM 出现 *emergent outliers***（Dettmers 2022, *LLM.int8()*）：

- **极少数 channel 幅值爆炸**：约 0.1% 的 channel，幅值是其余的 **20-100×**。
- **结构性持续**：这些大 channel **固定在同样的维度**上，跨所有 token、所有 sequence 都出现（不是随机噪声）。
- **系统性**：它们出现在残差流、attention、FFN 的特定投影里。

**为什么这击垮朴素量化？** per-tensor 量化用一个 scale 覆盖所有 channel：
$$\text{scale} = \frac{\max|X|}{127}$$
一个 outlier 把 $\max|X|$ 拉到 100×，于是其余 99.9% 的正常 channel 被压进 1-2 个 INT8 码值 → 精度崩塌。**这就是后续所有方法要解决的核心矛盾**。

本步你用 hook 测出每个 channel 的幅值分布，并判断哪些是 emergent outlier。

## 本步填空

两个 `logic` 函数，从零搭出"扫描离群点"能力：

1. **`per_channel_magnitudes(activations)`** —— 给定一批激活 `[batch, seq, hidden]`，算每个 channel（最后一维）的幅值统计。
2. **`is_emergent_outlier(mags, ratio_threshold)`** —— **判断型**：给定幅值数组与一个倍数阈值，返回哪些 channel 是 emergent outlier（幅值 ≥ 均值的 ratio_threshold 倍）。

填完跑 `%%ipytest`（L1）→ tiny 合成验证（L2）→ 真 7B 扫描（L3）。

In [ ]:
def per_channel_magnitudes(activations):
    """给定一批激活张量，返回每个 channel（最后一维）的幅值统计。

    参数
    ----
    activations : torch.Tensor，形状 [..., hidden]（常见 [batch, seq, hidden]）。
                  最后一个维度是 channel 维。每个 channel 我们要看它的"典型幅值"。

    返回
    ----
    torch.Tensor，形状 [hidden]，dtype float32。
      每个 channel 的"典型幅值"——用 **该 channel 所有元素的绝对值的均值**（mean(|x|)）。
      为什么用均值而不是 max：max 会被单个极端 token 主导，mean(|x|) 反映 channel
      *结构性*的活跃程度，更稳。

    提示
    ----
      - 先取绝对值（torch.abs）。
      - 在 channel 维以外的所有维度上求平均，把张量降到 [hidden]。
      - 用 mean 而不是 sum/max；返回值形状严格 [hidden]，dtype float32。
      - 坑：不要用 dim=-1 直接 mean——那样得到的是"每元素绝对值的均值"标量。
        你要在"除最后一维以外的所有维"上做 mean，保留最后一维。
        一种写法：abs 后 reshape 成 [N, hidden] 再对 dim=0 取 mean。
    """
    # TODO: 实现 per-channel 平均绝对幅值，返回形状 [hidden] 的 float32 张量。
    raise NotImplementedError


# 脚手架（提供）：注册 forward hook，自动收集所有 Linear 输入激活。
def collect_linear_activations(model, input_ids, layer_filter=None):
    """跑一次 forward，用 hook 收集所有（或 layer_filter 命名的）Linear 输入激活。

    返回 dict: {模块名: activations_tensor}。activations_tensor 形状 [batch, seq, hidden]。
    """
    acts = {}
    handles = []
    def make_hook(name):
        def hook(module, inp, out):
            x = inp[0]
            if x.dim() >= 2:
                acts[name] = x.detach().float().cpu()
        return hook
    for name, mod in model.named_modules():
        if isinstance(mod, nn.Linear):
            if layer_filter is None or layer_filter in name:
                handles.append(mod.register_forward_hook(make_hook(name)))
    with torch.no_grad():
        model(input_ids)
    for h in handles:
        h.remove()
    return acts

In [ ]:
def is_emergent_outlier(magnitudes, ratio_threshold=6.0):
    """判断哪些 channel 是 emergent outlier。

    定义（本课程采用，对齐 LLM.int8() 论文直觉）：一个 channel 是 emergent outlier，
    当且仅当它的幅值 >= **全体 channel 幅值均值的 ratio_threshold 倍**。

    参数
    ----
    magnitudes : torch.Tensor，形状 [hidden]（来自 per_channel_magnitudes）。
    ratio_threshold : float，默认 6.0。
      含义：幅值达到均值的几倍才算 outlier。6× 是一个"明显结构性突出"的经验值——
      论文里 emergent outlier 常达 20×+，但 6× 已足以把异常维度从噪声里分出来。

    返回
    ----
    torch.BoolTensor，形状 [hidden]，True 表示该 channel 是 emergent outlier。

    提示
    ----
      - 先算所有 channel 幅值的均值（标量）。
      - 判断型核心：阈值 = 均值 * ratio_threshold；返回 magnitudes >= 阈值 的布尔张量。
      - 思考（不是写代码）：若 ratio_threshold 调到 1.0，几乎所有 channel 都会被判为 outlier
        ——判断会失效。这正是"判断型空"的训练点：理解阈值语义，不是抄字面量。
    """
    # TODO: 返回布尔张量，标记每个 channel 是否 >= 均值的 ratio_threshold 倍。
    raise NotImplementedError

In [ ]:
%%ipytest -qq

def test_per_channel_magnitudes_shape_and_value():
    x = torch.tensor([[[1.0, -2.0, 3.0],
                       [3.0,  2.0, 1.0]]])   # [1,2,3]
    mags = per_channel_magnitudes(x)
    assert mags.shape == torch.Size([3])
    assert mags.dtype == torch.float32
    # channel0: mean(|1|,|3|)=2.0 ; channel1: mean(|-2|,|2|)=2.0 ; channel2: mean(|3|,|1|)=2.0
    assert torch.allclose(mags, torch.tensor([2.0, 2.0, 2.0]))

def test_per_channel_magnitudes_picks_outlier_channel():
    # channel 2 有明显 outlier（30），channel 0/1 小
    x = torch.tensor([[[1.0, 1.0, 30.0],
                       [1.0, 1.0, 30.0]]])
    mags = per_channel_magnitudes(x)
    assert mags[2].item() > mags[0].item() * 10   # 30 vs 1，远超 10×

def test_is_emergent_outlier_flags_outlier():
    mags = torch.tensor([1.0, 1.0, 1.0, 30.0])   # 均值 = 8.25
    out = is_emergent_outlier(mags, ratio_threshold=3.0)   # 阈值 = 8.25*3 = 24.75
    assert out.tolist() == [False, False, False, True]

def test_is_emergent_outlier_no_outlier_when_flat():
    mags = torch.tensor([2.0, 2.0, 2.0, 2.0])    # 均值 2，任何 ratio 都无 outlier
    out = is_emergent_outlier(mags, ratio_threshold=6.0)
    assert out.sum().item() == 0

def test_is_emergent_outlier_threshold_semantics():
    # 阈值 1.0 时，超过均值的都算 → 30 与 1 中只有 30 超过均值 8.25
    mags = torch.tensor([1.0, 30.0])
    out = is_emergent_outlier(mags, ratio_threshold=1.0)
    assert out.tolist() == [False, True]

## L2：tiny 合成验证（CPU）

合成一个带 *emergent outlier* 的激活张量（30× 的 channel），用你的函数检出它——确认逻辑对"结构性突出"敏感。再跑一个内存里随机初始化的 **tiny Qwen2**，抓它的 Linear 激活（虽无真 outlier，但验证 hook + magnitude 管线通）。

In [ ]:
# 合成验证：注入一个 30× outlier channel，确认能被检出
torch.manual_seed(0)
synth = torch.randn(2, 16, 64)            # [batch=2, seq=16, hidden=64]
synth[..., 7] *= 30.0                      # channel 7 设为 emergent outlier（30×）

mags = per_channel_magnitudes(synth)
outliers = is_emergent_outlier(mags, ratio_threshold=6.0)
print("合成激活 channel 幅值：均值 {:.3f}，最大 {:.3f}（在 channel {}）".format(
    mags.mean().item(), mags.max().item(), int(mags.argmax())))
print("检出 outlier channel 数:", int(outliers.sum().item()), "->", torch.where(outliers)[0].tolist())
assert bool(outliers[7]), "channel 7 应被检出为 outlier"
print("L2a PASS：合成 30× outlier 被正确检出")

# 用内存 tiny Qwen2 验证 hook + magnitude 管线（随机权重，无真 outlier，只验管线通）
from transformers import Qwen2Config, Qwen2ForCausalLM
def make_tiny_model(vocab_size=320, hidden_size=128):
    cfg = Qwen2Config(num_hidden_layers=2, hidden_size=hidden_size,
        intermediate_size=hidden_size*2, num_attention_heads=4,
        num_key_value_heads=2, vocab_size=vocab_size, tie_word_embeddings=True)
    return Qwen2ForCausalLM(cfg).eval()

tiny = make_tiny_model()
ids = torch.randint(0, 320, (1, 8))
acts = collect_linear_activations(tiny, ids, layer_filter="mlp.up_proj")
assert len(acts) >= 1, "应至少抓到一个 mlp.up_proj 的激活"
name = list(acts)[0]; x = acts[name]
print("\ntiny Qwen2 抓到激活:", name, "shape", tuple(x.shape))
mags_t = per_channel_magnitudes(x)
assert mags_t.shape[0] == x.shape[-1]
print("tiny 激活 channel 幅值：均值 {:.4f}，max/mean = {:.2f}×".format(
    mags_t.mean().item(), (mags_t.max()/mags_t.mean()).item()))
print("L2b PASS：hook + per_channel_magnitudes 管线在 tiny Qwen2 上跑通")

## L3：H200 执行（真 Qwen2.5-0.5B 再 7B，写 outlier_scan.json）

GPU 守卫：无 GPU 自动跳过。本机若是 H200/L20X 则真跑——扫描真模型的多个 Linear，看 emergent outlier。

> **数值锚（避免你怀疑实现错了）**：emergent outlier 随模型规模增强——**0.5B 上你看到的 max/mean 会明显小于原理里强调的"20×+"**，这是预期现象，不是 bug。本机实测的预期区间：
>
> | 模型 | gate/up_proj max/mean | q/o_proj max/mean |
> |------|----------------------|-------------------|
> | **0.5B**（本步必跑、显存够） | **~18×** | ~9-13× |
> | **7B**（显存够才跑） | **~58×** | ~17-21× |
>
> 判断标准不是"必须到 20×+"，而是 **`max_over_mean` 显著 > 6× 且 outlier channel 集中在极少数维度（< 2%）**——结构性地、固定在同样 channel 上出现，而非随机噪声。0.5B 已能满足这条（gate_proj 18×、1/896 outlier），7B 更戏剧（58×、1/3584）。若你的 0.5B 数字落在 9-18× 区间，**实现是对的**；7B 的 20×+ 是规模放大的结果。

In [ ]:
def run_outlier_scan(model_dir, save_path, sample_text="The quick brown fox jumps over the lazy dog. " * 4):
    """加载真模型，取一段文本 forward，抓若干 Linear 的输入激活，写 outlier 扫描结果到 JSON。"""
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForCausalLM.from_pretrained(model_dir, dtype=torch.float16,
                                                 device_map="auto").eval()
    ids = tok(sample_text, return_tensors="pt").input_ids.to(model.device)

    # 扫描几个有代表性的投影（FFN 的 gate/up、attention 的 q/k/v）
    targets = [".mlp.up_proj", ".mlp.gate_proj", ".self_attn.q_proj", ".self_attn.o_proj"]
    scan = {}
    acts = collect_linear_activations(model, ids)
    # 取第一个 transformer 层的这些投影
    for name, x in acts.items():
        if not any(t in name for t in targets) or "layers.0." not in name:
            continue
        mags = per_channel_magnitudes(x.cpu())
        out = is_emergent_outlier(mags, ratio_threshold=6.0)
        scan[name] = {
            "hidden": int(mags.shape[0]),
            "mean_magnitude": float(mags.mean()),
            "max_magnitude": float(mags.max()),
            "max_over_mean": float(mags.max() / max(mags.mean().item(), 1e-9)),
            "n_outliers": int(out.sum()),
            "outlier_ratio": float(out.float().mean()),
        }
    save_path = pathlib.Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)
    save_path.write_text(json.dumps(scan, indent=2, ensure_ascii=False))
    del model; torch.cuda.empty_cache()
    return scan

if torch.cuda.is_available():
    # 先 0.5B 快验
    scan_05b = run_outlier_scan(TINY_MODEL_DIR, OUT_ROOT / "outlier_scan_0.5B.json")
    print("0.5B outlier scan:")
    for k, v in scan_05b.items():
        print(f"  {k}: max/mean={v['max_over_mean']:.1f}×, outliers={v['n_outliers']}/{v['hidden']}")
    # 再 7B（emergent 现象更明显）
    scan_7b = run_outlier_scan(MODEL_DIR, OUT_ROOT / "outlier_scan_7B.json")
    print("\n7B outlier scan:")
    for k, v in scan_7b.items():
        print(f"  {k}: max/mean={v['max_over_mean']:.1f}×, outliers={v['n_outliers']}/{v['hidden']}")
else:
    print("跳过 L3：无 GPU（CPU 环境只跑 L1/L2）。")

## 产物检查

打印 L3 写出的 `outlier_scan_7B.json`，看真 7B 的 emergent outlier 幅值比。

In [ ]:
def report_scan(path):
    path = pathlib.Path(path)
    if not path.exists():
        print(f"(跳过：{path} 不存在，可能 L3 未跑)")
        return
    scan = json.loads(path.read_text())
    print(f"== {path.name} ==")
    for name, v in scan.items():
        bar = "█" * min(int(v["max_over_mean"]), 40)
        print(f"  {name}")
        print(f"    max/mean = {v['max_over_mean']:6.1f}×  {bar}")
        print(f"    outlier channel: {v['n_outliers']}/{v['hidden']} ({v['outlier_ratio']:.2%})")
    # 判断标准：不是"必须到 20×+"，而是 max/mean 显著 > 6× 且 outlier 集中在 <2% 的固定 channel。
    # 0.5B（~18×）已证实 emergent outlier 存在；7B（~58×）是规模放大、更戏剧——两者都说明现象成立。
    max_ratio = max(v["max_over_mean"] for v in scan.values()) if scan else 0
    print(f"\n  本模型最大 channel 幅值比: {max_ratio:.1f}×")
    if max_ratio > 6.0:
        print(f"  → > 6×，证实 emergent outlier 存在（结构性离群 channel）。")
        if "0.5B" in path.name:
            print(f"  注：这是 0.5B（~18×），小于原理强调的 20×+；7B 上会到 ~58×。")
            print(f"     emergent outlier 随规模增强——0.5B 满足判断标准即说明实现正确，不必追求 20×+。")
        else:
            print(f"  → >> 6×（7B 规模放大到 ~58×），emergent outlier 最戏剧的体现。")
    else:
        print(f"  注：max/mean ≤ 6×——若这是 0.5B 且显著低于 ~18× 预期，检查 per_channel_magnitudes 实现。")

report_scan(OUT_ROOT / "outlier_scan_7B.json")
report_scan(OUT_ROOT / "outlier_scan_0.5B.json")